In [1]:
import os
import random
import numpy as np
import pandas as pd
from scipy.ndimage import rotate
from sklearn.model_selection import train_test_split

# ── Paths ─────────────────────────────────────────────────────
COHORT_CSV   = "D:/mamba_model/thesis_cohort_final.csv"
PET_CACHE    = "D:/mamba_model/preprocessed_cache_pet"        # source (originals)
PET_AUG_DIR  = "D:/mamba_model/preprocessed_cache_pet_aug"    # output (orig + aug copies)

os.makedirs(PET_AUG_DIR, exist_ok=True)

AUG_SEEDS = [1, 101, 42]  # matches MRI augmentation seeds exactly

print(f"PET cache:     {PET_CACHE}")
print(f"Output dir:    {PET_AUG_DIR}")
print(f"Aug seeds:     {AUG_SEEDS}")


PET cache:     D:/mamba_model/preprocessed_cache_pet
Output dir:    D:/mamba_model/preprocessed_cache_pet_aug
Aug seeds:     [1, 101, 42]


In [3]:
# ── Recreate the same train/val/test split used for MRI ───────
# Must match MRI_Extraction.ipynb / mamba_cnn_hybrid_test.ipynb exactly,
# so the same subjects get augmented on both modalities.

df = pd.read_csv(COHORT_CSV)
sessions = df["mri_session"].values
labels   = df["outcome_label"].values

X_tv, X_test, y_tv, y_test = train_test_split(
    sessions, labels, test_size=0.2, random_state=42, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv
)

# Map mri_session -> subject_id (PET cache is keyed by subject_id, not session)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))
train_subject_ids = [session_to_subject[s] for s in X_train]

print(f"Total subjects:     {len(df)}")
print(f"Training subjects:  {len(train_subject_ids)}")
print(f"Val subjects:       {len(X_val)}")
print(f"Test subjects:      {len(X_test)}")

Total subjects:     210
Training subjects:  126
Val subjects:       42
Test subjects:      42


In [4]:
# ── Augmentation function — identical logic to augment_roi_simple (MRI) ─

def augment_roi_simple(roi_array, seed):
    """Same rotation + flip augmentation as the MRI pipeline."""
    np.random.seed(seed)
    random.seed(seed)

    augmented = roi_array.copy()

    # Random rotation
    angle = np.random.uniform(-15, 15)
    for i in range(roi_array.shape[0]):
        augmented[i] = rotate(
            augmented[i], angle, reshape=False, mode="constant", cval=0
        )

    # Random flips
    if np.random.random() > 0.5:
        augmented = augmented[:, ::-1, :, :].copy()
    if np.random.random() > 0.5:
        augmented = augmented[:, :, ::-1, :].copy()

    return augmented.astype(np.float32)


In [5]:
# Copy originals into the aug directory (all subjects) ──
# Matches the "_orig.npy" convention used by preprocessed_cache_roi64_aug/

print("Copying originals...")
copied = 0
missing = []
for _, row in df.iterrows():
    subject_id = row["subject_id"]
    src = f"{PET_CACHE}/{subject_id}_PIB.npy"
    dst = f"{PET_AUG_DIR}/{subject_id}_orig.npy"

    if os.path.exists(dst):
        continue
    if not os.path.exists(src):
        missing.append(subject_id)
        continue

    arr = np.load(src)
    np.save(dst, arr)
    copied += 1

print(f"Copied {copied} originals. Missing source files: {len(missing)}")
if missing:
    print("  Missing:", missing[:10], "..." if len(missing) > 10 else "")

Copying originals...
Copied 210 originals. Missing source files: 0


In [6]:
# Generate augmented versions for training subjects only ─

print("Generating augmented PET files...")
failed = []
for seed in AUG_SEEDS:
    for subject_id in train_subject_ids:
        save_path = f"{PET_AUG_DIR}/{subject_id}_aug{seed}.npy"
        if os.path.exists(save_path):
            continue
        try:
            orig = np.load(f"{PET_AUG_DIR}/{subject_id}_orig.npy")
            augmented = augment_roi_simple(orig, seed)
            np.save(save_path, augmented)
        except Exception as e:
            print(f"  FAILED {subject_id} seed {seed}: {e}")
            failed.append((subject_id, seed))
    print(f"  Seed {seed} done")

print(f"\nDone! Failed: {len(failed)}")
print(f"Total training files: {len(train_subject_ids)} orig + "
      f"{len(train_subject_ids) * len(AUG_SEEDS)} augmented "
      f"= {len(train_subject_ids) * (1 + len(AUG_SEEDS))}")

Generating augmented PET files...
  Seed 1 done
  Seed 101 done
  Seed 42 done

Done! Failed: 0
Total training files: 126 orig + 378 augmented = 504


In [7]:
# Verify

sample_orig = np.load(f"{PET_AUG_DIR}/{train_subject_ids[0]}_orig.npy")
sample_aug  = np.load(f"{PET_AUG_DIR}/{train_subject_ids[0]}_aug1.npy")

print("Verification:")
print(f"  Orig mean: {sample_orig.mean():.6f}")
print(f"  Aug1 mean: {sample_aug.mean():.6f}")
print(f"  Different: {not np.allclose(sample_orig, sample_aug)}")
print(f"  Mean abs difference: {np.abs(sample_orig - sample_aug).mean():.6f}")

Verification:
  Orig mean: -0.000047
  Aug1 mean: -0.000047
  Different: True
  Mean abs difference: 0.128958
